In [1]:
import duckdb
import time
from typing import Dict, Any
from pathlib import Path


In [2]:
def get_sandbox_connection(spider_db_path: Path | None = None):
    con = duckdb.connect(database=":memory:")

    if spider_db_path is not None:
        spider_db_path = spider_db_path.resolve()

        if not spider_db_path.exists():
            raise FileNotFoundError(spider_db_path)

        # Enable SQLite
        con.execute("INSTALL sqlite;")
        con.execute("LOAD sqlite;")

        # Attach Spider DB
        con.execute(f"""
        ATTACH DATABASE '{spider_db_path}' AS spider_db (TYPE sqlite);
        """)

        # 🔑 CRITICAL FIX — set search path
        con.execute("SET schema 'spider_db';")

    return con


In [3]:
def explain_cost(con: duckdb.DuckDBPyConnection, sql: str) -> str:
    try:
        plan = con.execute(f"EXPLAIN ANALYZE {sql}").fetchall()
        return "\n".join(str(p[0]) for p in plan)
    except Exception as e:
        raise RuntimeError(f"EXPLAIN failed: {e}")


In [4]:
def estimate_row_count(con: duckdb.DuckDBPyConnection, sql: str) -> int:
    try:
        sql_clean = sql.strip().rstrip(";")   # 🔥 FIX
        wrapped = f"SELECT COUNT(*) FROM ({sql_clean}) t"
        return con.execute(wrapped).fetchone()[0]
    except Exception as e:
        return -1


In [5]:
def execute_with_timeout(con, sql: str, timeout_sec: float = 2.0):
    sql_clean = sql.strip().rstrip(";")   # 🔥 FIX
    start = time.time()
    result = con.execute(sql_clean).fetchdf()
    elapsed = time.time() - start

    if elapsed > timeout_sec:
        raise TimeoutError(f"Query exceeded {timeout_sec}s")

    return result, elapsed


In [6]:
def dry_run_write(con, sql: str):
    try:
        con.execute("BEGIN TRANSACTION;")
        con.execute(sql)
        con.execute("ROLLBACK;")
        return True, None
    except Exception as e:
        con.execute("ROLLBACK;")
        return False, str(e)


In [7]:
def execution_verifier(
    sql: str,
    intent: str,
    db_path=None,
    max_rows: int = 100_000,
    timeout_sec: float = 2.0
):
    con = get_sandbox_connection(db_path)

    verdict = {
        "sql": sql,
        "intent": intent,
        "allowed": False,
        "reason": None,
        "rows": None,
        "latency": None,
        "plan": None
    }

    # ---- COST GATING ----
    try:
        verdict["plan"] = explain_cost(con, sql)
    except Exception as e:
        verdict["reason"] = str(e)
        return verdict

    # ---- WRITE QUERIES ----
    if intent == "WRITE":
        ok, err = dry_run_write(con, sql)
        if not ok:
            verdict["reason"] = f"WRITE dry-run failed: {err}"
            return verdict

        verdict["allowed"] = True
        verdict["reason"] = "WRITE dry-run successful"
        return verdict

    # ---- READ QUERIES ----
    row_est = estimate_row_count(con, sql)
    if row_est == -1:
        verdict["reason"] = "Row count estimation failed"
        return verdict

    if row_est > max_rows:
        verdict["reason"] = f"Row limit exceeded: {row_est}"
        return verdict

    try:
        df, latency = execute_with_timeout(con, sql, timeout_sec)
        verdict["allowed"] = True
        verdict["rows"] = len(df)
        verdict["latency"] = latency
        verdict["reason"] = "Execution successful"
        return verdict
    except Exception as e:
        verdict["reason"] = str(e)
        return verdict


In [8]:
sql = """
SELECT a.name, p.title
FROM author a
JOIN writes w ON a.aid = w.aid
JOIN publication p ON w.pid = p.pid
LIMIT 5;
"""

intent = "READ"

result = execution_verifier(
    sql,
    intent,
    db_path=Path("../data/spider/database/academic/academic.sqlite")
)

for k, v in result.items():
    print(f"{k}: {v}")


sql: 
SELECT a.name, p.title
FROM author a
JOIN writes w ON a.aid = w.aid
JOIN publication p ON w.pid = p.pid
LIMIT 5;

intent: READ
allowed: True
reason: Execution successful
rows: 0
latency: 0.7681646347045898
plan: analyzed_plan


In [9]:
sql = """
SELECT a.name, p.title
FROM author a
JOIN writes w ON a.aid = w.aid
JOIN publication p ON w.pid = p.pid
LIMIT 5;
"""

intent = "WRITE"

result = execution_verifier(
    sql,
    intent,
    db_path=Path("../data/spider/database/academic/academic.sqlite")
)

for k, v in result.items():
    print(f"{k}: {v}")


sql: 
SELECT a.name, p.title
FROM author a
JOIN writes w ON a.aid = w.aid
JOIN publication p ON w.pid = p.pid
LIMIT 5;

intent: WRITE
allowed: True
reason: WRITE dry-run successful
rows: None
latency: None
plan: analyzed_plan
